# Lesson 04 - Дизайнерски шаблон за използване на инструменти

В този урок ще научите за дизайна на шаблона **Използване на инструменти** за AI агенти с помощта на Microsoft Agent Framework (Python). Обсъждаме:

- Дефиниране на функционални инструменти с декоратора `@tool` и типизирани параметри
- Осигуряване на схеми на инструментите, за да знае моделът какво прави всеки инструмент
- Контролиране на изпълнението на инструменти с `approval_mode`
- Връщане на **структуриран изход** чрез модели Pydantic и `response_format`

Сценарият е **агент за резервации за пътувания**, който може да търси дестинации, да проверява наличности и да извлича информация за полети.


## Настройка


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Дефиниране на инструменти с декоратора @tool

Декораторът `@tool` превръща обикновена Python функция в инструмент, който агентът може да извиква.
Основни точки:

- **Докстрингът** става описание на инструмента, което моделът вижда.
- **Анотациите на типове** (включително `Annotated` с описания) определят схемата на инструмента.
- `approval_mode` контролира дали потребителят трябва да одобри всяко извикване преди то да бъде изпълнено.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Създаване на агент с множество инструменти

Пратете всички три инструменти на клиента, така че моделът да може да използва тези, от които има нужда, за да отговори на въпроса на потребителя.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Структуриран изход с инструменти

Като зададете `response_format` на Pydantic модел, агентът е принуден да върне добре типизиран JSON обект, вместо свободен текст. Това е полезно, когато последващият код трябва да обработва резултата програмно.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Модели за одобрение на инструменти

Параметърът `approval_mode` на `@tool` контролира дали повикванията на инструмента изискват човешко одобрение преди изпълнение:

| Режим | Поведение |
|---|---|
| `"never_require"` | Инструментът се изпълнява автоматично — не е необходимо потвърждение от потребителя. |
| `"always_require"` | Всяко повикване трябва да бъде одобрено от потребителя преди изпълнението. |

Използвайте `"always_require"` за инструменти, които имат странични ефекти (например резервиране на полет, таксуване на кредитна карта), за да може човек да остане във веригата.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Обобщение

В този урок научихте как да:

1. **Дефинирате инструменти** с помощта на декоратора `@tool` с типизирани параметри и docstrings, които служат като схема на инструмента.
2. **Комбинирате няколко инструмента**, така че агентът да може да ги извиква последователно за отговор на сложни заявки.
3. **Връщате структурирани резултати**, като подавате Pydantic модел като `response_format`.
4. **Контролирате одобрението на инструменти** с `approval_mode`, за да поддържате човешки контрол при чувствителни операции.

Тези модели формират основата за изграждане на надеждни, готови за производство агенти, които могат безопасно да взаимодействат с външни системи.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с използване на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за никакви недоразумения или погрешни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
